In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7"

import logging
import warnings
from pathlib import Path

import pandas as pd
import torch
import torch.distributed as dist
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers

# Suppress specific warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*custom_fwd.*")
warnings.filterwarnings("ignore", message=".*custom_bwd.*")
warnings.filterwarnings("ignore", message=".*evaluation_strategy.*")

# Set up logging
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[logging.FileHandler("training.log"), logging.StreamHandler()],
)


def load_and_validate_data(file_path: Path) -> Dataset:
    """Load and validate dataset with cancer type information"""
    logging.info(f"Loading data from {file_path}")
    df = pd.read_csv(file_path)

    # Validate data
    required_columns = ["anchor", "positive", "cancer_type"]
    assert all(col in df.columns for col in required_columns), (
        f"Missing required columns: {required_columns}"
    )

    unique_cancer_types = df["cancer_type"].unique()

    # Log dataset statistics
    logging.info(f"Dataset size: {len(df)}")
    logging.info(f"Number of cancer types: {len(unique_cancer_types)}")
    logging.info(
        f"Cancer types distribution:\n{df['cancer_type'].value_counts(normalize=True)}"
    )

    return Dataset.from_dict(
        {
            "sentence1": df["anchor"].tolist(),
            "sentence2": df["positive"].tolist(),
        }
    )


def setup_ddp():
    """Setup distributed training"""
    if "WORLD_SIZE" in os.environ:
        world_size = int(os.environ["WORLD_SIZE"])
        rank = int(os.environ["LOCAL_RANK"])
    else:
        world_size = torch.cuda.device_count()
        rank = int(os.environ.get("LOCAL_RANK", 0))

    if world_size > 1:
        dist.init_process_group("nccl")
        torch.cuda.set_device(rank)

    return rank, world_size


def main():
    # Setup DDP
    rank, world_size = setup_ddp()
    is_main_process = rank == 0

    # Define paths and create output directory
    output_dir = Path("models") / "mnrl_full_data"
    if is_main_process:
        output_dir.mkdir(parents=True, exist_ok=True)

    # Training configuration
    config = {
        "model_name": "jinaai/jina-embeddings-v3",
        "num_epochs": 10,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "gradient_accumulation_steps": 8,
        "learning_rate": 1e-5,
        "warmup_ratio": 0.15,
        "max_grad_norm": 1.0,
    }

    # Load datasets
    train_dataset = load_and_validate_data("data/training/mnrl/train.csv")
    eval_dataset = load_and_validate_data("data/training/mnrl/eval.csv")

    model = SentenceTransformer(
        config["model_name"],
        trust_remote_code=True,
        model_kwargs={
            "default_task": "text-matching",
            "lora_main_params_trainable": True,
        },
        model_card_data=SentenceTransformerModelCardData(
            language=["en"],
            license="apache-2.0",
            model_name="Jina-v3-Clinical-Cross-Modal-Alignment",
            model_id="Jina-v3-Clinical-Cross-Modal-Alignment",
            tags=["clinical", "radiology", "pathology", "cross-modal"],
        ),
    )

    total_steps_per_epoch = len(train_dataset) // (
        config["train_batch_size"] * config["gradient_accumulation_steps"] * world_size
    )

    if is_main_process:
        logging.info(f"Total steps per epoch: {total_steps_per_epoch}")

    # Training arguments
    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=config["num_epochs"],
        per_device_train_batch_size=config["train_batch_size"],
        per_device_eval_batch_size=config["eval_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        learning_rate=config["learning_rate"],
        warmup_ratio=config["warmup_ratio"],
        max_grad_norm=config["max_grad_norm"],
        lr_scheduler_type="cosine",
        bf16=True,
        eval_strategy="steps",
        eval_steps=total_steps_per_epoch,
        save_steps=total_steps_per_epoch,
        save_total_limit=2,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=100,
        logging_dir=str(output_dir / "logs"),
        local_rank=rank,
        ddp_backend="nccl",
        dataloader_drop_last=True,
        gradient_checkpointing=True,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        report_to="none",
    )

    # Initialize loss
    train_loss = losses.MultipleNegativesRankingLoss(model)

    # Initialize trainer without evaluator
    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=train_loss,
        evaluator=None,
    )

    # Train
    if is_main_process:
        logging.info("Starting training...")
    trainer.train()

    # Save final model and results only on main process
    if is_main_process:
        final_output_dir = output_dir / "final"
        model.save(str(final_output_dir))

        logging.info(f"Model saved to {final_output_dir}")

    # Cleanup
    if world_size > 1:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()
